In [1]:
import pandas as pd
import numpy as np

In [2]:
# 1. Load the raw datasets
customers = pd.read_csv("C:/Users/DELL/Downloads/Emergence Assignment/Data Analytics Raw Data-20260123T082005Z-3-001/Data Analytics Raw Data/raw data/customers.csv")
subscriptions = pd.read_csv("C:/Users/DELL/Downloads/Emergence Assignment/Data Analytics Raw Data-20260123T082005Z-3-001/Data Analytics Raw Data/raw data/subscriptions.csv")
events = pd.read_csv("C:/Users/DELL/Downloads/Emergence Assignment/Data Analytics Raw Data-20260123T082005Z-3-001/Data Analytics Raw Data/raw data/events.csv")

In [3]:
# 2. Check shapes and missing values
print("--- CUSTOMERS ---")
print(customers.head())
print(customers.info())
print(customers.isnull().sum())

--- CUSTOMERS ---
  customer_id signup_date     segment country  is_enterprise
0       C0001  2023-01-04  Enterprise      IN           True
1       C0002  2023-01-14         SMB      CA          False
2       C0003  2023-01-04         SMB      IN          False
3       C0004  2023-03-19         SMB      CA          False
4       C0005  2023-03-25         NaN      IN          False
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   customer_id    1000 non-null   str  
 1   signup_date    964 non-null    str  
 2   segment        757 non-null    str  
 3   country        1000 non-null   str  
 4   is_enterprise  1000 non-null   bool 
dtypes: bool(1), str(4)
memory usage: 32.4 KB
None
customer_id        0
signup_date       36
segment          243
country            0
is_enterprise      0
dtype: int64


In [4]:
print("\n--- SUBSCRIPTIONS ---")
print(subscriptions.head())
print(subscriptions.info())
print(subscriptions.isnull().sum())


--- SUBSCRIPTIONS ---
  subscription_id customer_id  start_date    end_date  monthly_price    status
0          S00001       C0001  2023-01-10         NaN            499    active
1          S00002       C0002  2023-01-12         NaN             49    active
2          S00003       C0003  2023-01-08  2023-02-14            149  canceled
3          S00004       C0004  2023-01-21         NaN            149    active
4          S00005       C0005  2023-04-03         NaN             99    active
<class 'pandas.DataFrame'>
RangeIndex: 941 entries, 0 to 940
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   subscription_id  941 non-null    str  
 1   customer_id      941 non-null    str  
 2   start_date       941 non-null    str  
 3   end_date         223 non-null    str  
 4   monthly_price    941 non-null    int64
 5   status           941 non-null    str  
dtypes: int64(1), str(5)
memory usage: 44.2 KB
None
subscr

In [5]:
print("\n--- EVENTS ---")
print(events.head())
print(events.info())
print(events.isnull().sum())


--- EVENTS ---
  event_id customer_id   event_type  event_date    source
0  E000001       C0001       signup  2023-03-10   organic
1  E000002       C0001       signup  2023-03-10   organic
2  E000003       C0002       signup  2023-03-27       ads
3  E000004       C0002  trial_start  2023-03-28       ads
4  E000005       C0003       signup  2023-02-19  referral
<class 'pandas.DataFrame'>
RangeIndex: 2411 entries, 0 to 2410
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   event_id     2411 non-null   str  
 1   customer_id  2411 non-null   str  
 2   event_type   2411 non-null   str  
 3   event_date   2411 non-null   str  
 4   source       2411 non-null   str  
dtypes: str(5)
memory usage: 94.3 KB
None
event_id       0
customer_id    0
event_type     0
event_date     0
source         0
dtype: int64


In [6]:
# 3. Check for duplicates
print("\n--- DUPLICATE CHECKS ---")
print(f"Customer duplicates (by customer_id): {customers['customer_id'].duplicated().sum()}")
print(f"Subscription duplicates (by subscription_id): {subscriptions['subscription_id'].duplicated().sum()}")
print(f"Subscription duplicates (by customer_id): {subscriptions['customer_id'].duplicated().sum()}")
print(f"Event duplicates (by event_id): {events['event_id'].duplicated().sum()}")



--- DUPLICATE CHECKS ---
Customer duplicates (by customer_id): 0
Subscription duplicates (by subscription_id): 0
Subscription duplicates (by customer_id): 35
Event duplicates (by event_id): 0


In [7]:
# Investigate how the missing segment subset of customers
customers[customers["segment"].isna()]

,customer_id,signup_date,segment,country,is_enterprise
4,C0005,2023-03-25,NaN,IN,False
9,C0010,2023-01-06,NaN,CA,False
23,C0024,2023-02-20,NaN,IN,False
25,C0026,2023-03-16,NaN,UK,False
26,C0027,2023-01-18,NaN,US,False
...,...,...,...,...,...
968,C0969,2023-01-29,NaN,AU,False
973,C0974,2023-02-09,NaN,UK,False
978,C0979,2023-02-08,NaN,US,False
980,C0981,2023-03-14,NaN,IN,False


In [8]:
# check for inconsistencies in date
subscriptions[(subscriptions["start_date"] > subscriptions["end_date"])]

,subscription_id,customer_id,start_date,end_date,monthly_price,status


- Rule 1: Trust subscriptions.`start_date` as the financial source of truth.
- Rule 2: The "`True Signup Date`" for a customer is the earliest of their profile signup_date, subscription start_date, or first signup event date. 
- This guarantees that True `Signup <= Subscription Start`.


subscriptions[subscriptions[["customer_id","start_date"]].duplicated(keep=False)].sort_values(by=["customer_id"])

## Step 1: Clean the Customers Table

In [9]:
# 1. Normalize empty strings, whitespace, and case issues to true NaNs
customers['segment'] = customers['segment'].replace(r'^\s*$', np.nan, regex=True)

# 2. Audit and log the impact before mutating
missing_segments = customers['segment'].isna().sum()
print(f"[AUDIT] Imputing {missing_segments} missing segment values with 'Unknown'.")

# 3. Clean, fill, and optimize data type using method chaining
customers = customers.assign(
    segment=lambda df: df['segment']
    .fillna('Unknown')
    .str.strip()
    .astype('category')  # Optimize memory and query performance
)

[AUDIT] Imputing 243 missing segment values with 'Unknown'.


## Step 2: De-duplicate the Subscriptions Table

In [10]:
# Convert start_date to datetime first
subscriptions['start_date'] = pd.to_datetime(subscriptions['start_date'])
subscriptions['end_date'] = pd.to_datetime(subscriptions['end_date'])

# Drop duplicates on our logical key
subscriptions_clean = subscriptions.drop_duplicates(subset=['customer_id', 'start_date'], keep='first')


## Step 3: Extract True Signups from Events and Align Chronology


In [11]:
# Convert event_date to datetime
events['event_date'] = pd.to_datetime(events['event_date'])

In [12]:
# 1. Get first signup event per customer
first_signup_event = events[events['event_type'] == 'signup'].groupby('customer_id')['event_date'].min().reset_index()
first_signup_event.rename(columns={'event_date': 'first_signup_event_date'}, inplace=True)

In [13]:
# 2. Get first subscription start date per customer
first_sub_date = subscriptions_clean.groupby('customer_id')['start_date'].min().reset_index()
first_sub_date.rename(columns={'start_date': 'first_sub_start_date'}, inplace=True)

# 3. Merge these back to the customers table
customers_clean = customers.merge(first_signup_event, on='customer_id', how='left')
customers_clean = customers_clean.merge(first_sub_date, on='customer_id', how='left')
customers_clean["signup_date"] = pd.to_datetime(customers_clean["signup_date"])
# 4. Calculate the true signup date
customers_clean['true_signup_date'] = customers_clean[
    ['signup_date', 'first_signup_event_date', 'first_sub_start_date']
].min(axis=1)

# Drop temporary columns we used for calculation
customers_clean.drop(columns=['signup_date', 'first_signup_event_date', 'first_sub_start_date'], inplace=True)

In [14]:
events_clean = events.drop_duplicates(subset=["customer_id","event_date","event_type"],keep="first")

In [15]:
events_clean

,event_id,customer_id,event_type,event_date,source
0,E000001,C0001,signup,2023-03-10,organic
2,E000003,C0002,signup,2023-03-27,ads
3,E000004,C0002,trial_start,2023-03-28,ads
4,E000005,C0003,signup,2023-02-19,referral
5,E000006,C0004,signup,2023-01-16,ads
...,...,...,...,...,...
2406,E002407,C0999,trial_start,2023-03-15,organic
2407,E002408,C0999,activated,2023-03-17,organic
2408,E002409,C1000,signup,2023-01-10,referral
2409,E002410,C1000,trial_start,2023-01-11,referral


In [16]:
events_pivoted = (
    events_clean
    .pivot_table(
        index="customer_id",
        columns="event_type",
        values="event_date",
        aggfunc="min"  # Safely takes the earliest date if duplicates exist
    )
    .reset_index()      # Convert 'customer_id' from index back to a column
)
# Clean up column index names for clean merging later
events_pivoted.columns.name = None

In [22]:
# 1. Drop the redundant 'signup' event column, as we use 'true_signup_date'
events_for_merge = events_pivoted.drop(columns=['signup'], errors='ignore')

# 2. Perform defensive, explicit left joins
master_data = (
    customers_clean
    .merge(subscriptions_clean, on="customer_id", how="left")
    .merge(events_for_merge, on="customer_id", how="left")
)


In [24]:
master_data

,customer_id,segment,country,is_enterprise,true_signup_date,subscription_id,start_date,end_date,monthly_price,status,activated,churned,trial_start
0,C0001,Enterprise,IN,True,2023-01-04,S00001,2023-01-10,NaT,499.0,active,NaT,NaT,NaT
1,C0002,SMB,CA,False,2023-01-12,S00002,2023-01-12,NaT,49.0,active,NaT,NaT,2023-03-28
2,C0003,SMB,IN,False,2023-01-04,S00003,2023-01-08,2023-02-14,149.0,canceled,NaT,NaT,NaT
3,C0004,SMB,CA,False,2023-01-16,S00004,2023-01-21,NaT,149.0,active,NaT,NaT,NaT
4,C0005,Unknown,IN,False,2023-01-03,S00005,2023-04-03,NaT,99.0,active,NaT,2023-04-21,2023-01-04
...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,C0996,Enterprise,DE,True,2023-01-18,S00938,2023-04-06,NaT,79.0,active,NaT,NaT,2023-03-05
996,C0997,Mid-Market,DE,False,2023-01-10,S00939,2023-02-01,NaT,299.0,active,NaT,NaT,NaT
997,C0998,Mid-Market,AU,False,2023-01-02,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaT
998,C0999,SMB,CA,False,2023-01-06,S00940,2023-01-06,NaT,49.0,active,2023-03-17,NaT,2023-03-15


In [25]:

# 1. Define the conditional logic for Lifecycle Stages
conditions = [
    # Churned: Subscription is canceled OR they have a logged churned date
    (master_data['status'] == 'canceled') | (master_data['churned'].notna()),
    
    # Paid: Has subscription and it is active
    (master_data['subscription_id'].notna()) & (master_data['status'] == 'active'),
    
    # Activated: No subscription, but they have an activation event date
    (master_data['subscription_id'].isna()) & (master_data['activated'].notna()),
    
    # Trial: No subscription, no activation, but they started a trial
    (master_data['subscription_id'].isna()) & (master_data['trial_start'].notna()),
    
    # Signup: No subscription, no activation, no trial, but they signed up
    (master_data['subscription_id'].isna()) & (master_data['true_signup_date'].notna())
]

choices = ['Churned', 'Paid', 'Activated', 'Trial', 'Signup']

# Apply the conditions
master_data['life_cycle_stage'] = np.select(conditions, choices, default='Unknown')


# 2. Clean inconsistencies in the subscription end date (as done in SQL Task 2)
# If a customer has a subscription and they churned, ensure the end_date matches the churn event date
master_data['subscription_end_date'] = np.where(
    (master_data['subscription_id'].notna()) & (master_data['churned'].notna()),
    master_data['churned'],
    master_data['end_date']
)

# Rename columns for clarity and consistency
master_data = master_data.rename(columns={
    'start_date': 'subscription_start_date',
    'status': 'subscription_status'
})


In [26]:
master_data

,customer_id,segment,country,is_enterprise,true_signup_date,subscription_id,subscription_start_date,end_date,monthly_price,subscription_status,activated,churned,trial_start,life_cycle_stage,subscription_end_date
0,C0001,Enterprise,IN,True,2023-01-04,S00001,2023-01-10,NaT,499.0,active,NaT,NaT,NaT,Paid,NaT
1,C0002,SMB,CA,False,2023-01-12,S00002,2023-01-12,NaT,49.0,active,NaT,NaT,2023-03-28,Paid,NaT
2,C0003,SMB,IN,False,2023-01-04,S00003,2023-01-08,2023-02-14,149.0,canceled,NaT,NaT,NaT,Churned,2023-02-14
3,C0004,SMB,CA,False,2023-01-16,S00004,2023-01-21,NaT,149.0,active,NaT,NaT,NaT,Paid,NaT
4,C0005,Unknown,IN,False,2023-01-03,S00005,2023-04-03,NaT,99.0,active,NaT,2023-04-21,2023-01-04,Churned,2023-04-21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,C0996,Enterprise,DE,True,2023-01-18,S00938,2023-04-06,NaT,79.0,active,NaT,NaT,2023-03-05,Paid,NaT
996,C0997,Mid-Market,DE,False,2023-01-10,S00939,2023-02-01,NaT,299.0,active,NaT,NaT,NaT,Paid,NaT
997,C0998,Mid-Market,AU,False,2023-01-02,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaT,Signup,NaT
998,C0999,SMB,CA,False,2023-01-06,S00940,2023-01-06,NaT,49.0,active,2023-03-17,NaT,2023-03-15,Paid,NaT


In [29]:
cleaned_master_table = master_data.to_csv("cleaned_master_table.csv",index=False)

master_data['life_cycle_stage'].value_counts()

life_cycle_stage
Paid         525
Churned      407
Signup        29
Activated     20
Trial         19
Name: count, dtype: int64